# 05_silver_copernicus_marine.ipynb — Limpieza Copernicus Marine Bronze → Silver

Este notebook procesa:

```text
data/Copernicus Marine/*.nc
data/Copernicus Marine physics analysis/*.nc
```

y genera:

```text
silver/ocean_hourly/source=COPERNICUS_MARINE_WAVES/...
silver/ocean_physics/source=COPERNICUS_MARINE_PHYSICS/...
silver/tide_hourly/source=COPERNICUS_MARINE_PHYSICS_ZOS/...
```

Fuentes esperadas:

- **Copernicus Marine waves**: oleaje global a 3h.
- **Copernicus Marine physics analysis**: corrientes, temperatura, salinidad y `zos` diario.

Notas:
- El oleaje de 3h se interpola a 1h con `flag = 3` en valores interpolados.
- La física oceánica se conserva con `temporal_resolution = daily`.
- `zos` se guarda también como `sea_level` en `tide_hourly` con resolución diaria.

## Celda 0 — Montar Google Drive

In [17]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [18]:
!pip -q install xarray netCDF4 h5netcdf dask geopandas pyarrow shapely fiona tqdm scipy

## Celda 2 — Imports, rutas y configuración

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import re
import unicodedata
import json
import shutil
import gc
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
BRONZE_DIR = BASE_DIR / "data/bronze"
SILVER_DIR = BASE_DIR / "silver"

WAVES_DIR = BRONZE_DIR / "Copernicus Marine"
PHYSICS_DIR = BRONZE_DIR / "Copernicus Marine physics analysis"
DIM_ZONE_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

OUT_OCEAN_DIR = SILVER_DIR / "ocean_hourly"
OUT_PHYSICS_DIR = SILVER_DIR / "ocean_physics"
OUT_TIDE_DIR = SILVER_DIR / "tide_hourly"

QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

for d in [OUT_OCEAN_DIR, OUT_PHYSICS_DIR, OUT_TIDE_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_WAVES = "COPERNICUS_MARINE_WAVES"
SOURCE_PHYSICS = "COPERNICUS_MARINE_PHYSICS"
SOURCE_ZOS = "COPERNICUS_MARINE_PHYSICS_ZOS"

BBOX_CANARIAS = {
    "lat_min": 27.0,
    "lat_max": 29.5,
    "lon_min": -18.5,
    "lon_max": -13.0,
}

MIN_VALID_TS = pd.Timestamp("1999-01-01", tz="UTC")
MAX_VALID_TS = pd.Timestamp("2031-01-01", tz="UTC")

# Para pruebas rápidas: pon, por ejemplo, 1. Para procesar todo, deja None.
MAX_WAVE_FILES_FOR_TEST = None
MAX_PHYSICS_FILES_FOR_TEST = None
MAX_YEARS_FOR_TEST = None

print("WAVES_DIR existe:", WAVES_DIR.exists())
print("PHYSICS_DIR existe:", PHYSICS_DIR.exists())
print("DIM_ZONE_PATH existe:", DIM_ZONE_PATH.exists())

if not DIM_ZONE_PATH.exists():
    raise FileNotFoundError("No existe beach_geography.parquet. Ejecuta primero 01_silver_dim_zone.ipynb.")

WAVES_DIR existe: True
PHYSICS_DIR existe: True
DIM_ZONE_PATH existe: True


## Celda 3 — Utilidades generales

In [20]:
def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"\s+", " ", value)
    return value.upper()


def normalize_col(col):
    col = normalize_text(col)
    if pd.isna(col):
        return ""
    col = re.sub(r"[^A-Z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col


def standardize_longitudes_to_180(lon_values):
    lon = np.asarray(lon_values)
    return ((lon + 180) % 360) - 180


def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def make_point_id(prefix, lat, lon):
    lat_s = pd.Series(lat).round(4).astype(str)
    lon_s = pd.Series(lon).round(4).astype(str)
    return prefix + "_" + lat_s + "_" + lon_s


def current_speed_from_uv(u, v):
    return np.sqrt(u ** 2 + v ** 2)


def current_direction_from_uv(u, v):
    # Dirección hacia donde fluye la corriente, grados desde el norte.
    return (np.degrees(np.arctan2(u, v)) + 360) % 360


def remove_existing_source_partition(base_dir, source_name):
    source_path = base_dir / f"source={source_name}"
    if source_path.exists():
        shutil.rmtree(source_path)
        print("Eliminada partición antigua:", source_path)


def write_partitioned_parquet(df, base_dir):
    if df.empty:
        print("Chunk vacío. No se guarda.")
        return

    df = df.copy()
    df["source"] = df["source"].astype(str)
    df["isla"] = df["isla"].fillna("ISLA_DESCONOCIDA").astype(str)
    df["year"] = df["year"].astype("int64")

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(base_dir),
        partition_cols=["source", "year", "isla"],
        compression="snappy",
    )


def dataset_count_and_sample(path, source_name, sample_n=5):
    if not path.exists():
        return 0, pd.DataFrame()

    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    count = dataset.count_rows(filter=(ds.field("source") == source_name))

    if count == 0:
        return 0, pd.DataFrame()

    sample = dataset.head(sample_n, filter=(ds.field("source") == source_name)).to_pandas()

    return count, sample


def validate_timestamp_range(df, table_name, filename):
    if df.empty:
        return

    if df["timestamp"].isna().any():
        raise ValueError(f"{table_name} {filename}: timestamps nulos.")

    invalid = ~df["timestamp"].between(MIN_VALID_TS, MAX_VALID_TS)

    if invalid.any():
        raise ValueError(
            f"{table_name} {filename}: timestamps fuera de rango "
            f"{df.loc[invalid, 'timestamp'].min()} - {df.loc[invalid, 'timestamp'].max()}"
        )


def missing_pct(df, col):
    if df.empty or col not in df.columns:
        return 100.0
    return float(df[col].isna().mean() * 100)

## Celda 4 — Cargar `beach_geography`

In [21]:
beach_geography = pd.read_parquet(DIM_ZONE_PATH)

required_zone_cols = ["zona_id", "nombre_zona", "isla", "municipio", "lat", "lon"]
missing_zone_cols = [c for c in required_zone_cols if c not in beach_geography.columns]

if missing_zone_cols:
    raise ValueError(f"Faltan columnas en beach_geography: {missing_zone_cols}")

print("beach_geography shape:", beach_geography.shape)
display(beach_geography.head())

gdf_zones = gpd.GeoDataFrame(
    beach_geography.copy(),
    geometry=gpd.points_from_xy(beach_geography["lon"], beach_geography["lat"]),
    crs="EPSG:4326",
)

gdf_zones_m = gdf_zones.to_crs("EPSG:3857")

beach_geography shape: (561, 17)


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


## Celda 5 — Localizar archivos Copernicus Marine

In [22]:
wave_files = sorted(WAVES_DIR.glob("*.nc")) if WAVES_DIR.exists() else []
physics_files = sorted(PHYSICS_DIR.glob("*.nc")) if PHYSICS_DIR.exists() else []

if MAX_WAVE_FILES_FOR_TEST is not None:
    wave_files = wave_files[:MAX_WAVE_FILES_FOR_TEST]

if MAX_PHYSICS_FILES_FOR_TEST is not None:
    physics_files = physics_files[:MAX_PHYSICS_FILES_FOR_TEST]

print("Archivos waves:", len(wave_files))
for p in wave_files:
    print("-", p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

print("\nArchivos physics:", len(physics_files))
for p in physics_files:
    print("-", p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

if not wave_files:
    print("AVISO: no se encontraron archivos Copernicus Marine waves.")

if not physics_files:
    print("AVISO: no se encontraron archivos Copernicus Marine physics.")

Archivos waves: 3
- cmems_mod_glo_wav_my_0.2deg_PT3H-i_1778173932870.nc 659.44 MB
- cmems_mod_glo_wav_my_0.2deg_PT3H-i_1778174054080.nc 659.62 MB
- cmems_mod_glo_wav_my_0.2deg_PT3H-i_1778174103038.nc 340.31 MB

Archivos physics: 1
- cmems_mod_glo_phy_my_0.083deg_P1D-m_1778177336203.nc 521.36 MB


## Celda 6 — Funciones robustas para abrir y estandarizar NetCDF

In [23]:
WAVE_VAR_ALIASES = {
    "hs": ["VHM0", "vhm0", "swh", "SWH", "significant_wave_height"],
    "tp": ["VTPK", "vtpk", "peak_wave_period", "VTPK_SW1"],
    "wave_direction": ["VMDR", "vmdr", "mwd", "MWD", "mean_wave_direction"],
    "swell_height": ["VHM0_SW1", "vhm0_sw1", "VHM0_SW", "swell_height"],
    "swell_period": ["VTPK_SW1", "vtpk_sw1", "VTPK_SW", "swell_period"],
    "swell_direction": ["VMDR_SW1", "vmdr_sw1", "VMDR_SW", "swell_direction"],
    "wind_wave_height": ["VHM0_WW", "vhm0_ww", "wind_wave_height"],
    "wind_wave_period": ["VTPK_WW", "vtpk_ww", "wind_wave_period"],
    "stokes_u": ["STOKES_U", "stokes_u", "VSDX", "vsdx"],
    "stokes_v": ["STOKES_V", "stokes_v", "VSDY", "vsdy"],
}

PHYSICS_VAR_ALIASES = {
    "current_u": ["uo", "UO", "eastward_sea_water_velocity"],
    "current_v": ["vo", "VO", "northward_sea_water_velocity"],
    "sea_surface_temperature": ["thetao", "THETAO", "sohefldo", "temperature", "sea_water_potential_temperature"],
    "sea_surface_salinity": ["so", "SO", "salinity", "sea_water_salinity"],
    "zos": ["zos", "ZOS", "sea_surface_height_above_geoid", "sea_surface_height"],
}


def open_dataset_robust(path):
    last_error = None

    for engine in ["netcdf4", "h5netcdf", None]:
        try:
            if engine is None:
                return xr.open_dataset(path, chunks={})
            return xr.open_dataset(path, engine=engine, chunks={})
        except Exception as e:
            last_error = e

    raise ValueError(f"No se pudo abrir {path.name}: {repr(last_error)}")


def find_coord_name(ds_in, candidates):
    for c in candidates:
        if c in ds_in.coords or c in ds_in.dims or c in ds_in.variables:
            return c
    return None


def find_var_name(ds_in, aliases):
    available = set(ds_in.data_vars) | set(ds_in.variables)

    for alias in aliases:
        if alias in available:
            return alias

    norm_map = {normalize_col(v): v for v in available}

    for alias in aliases:
        alias_norm = normalize_col(alias)
        if alias_norm in norm_map:
            return norm_map[alias_norm]

    return None


def rename_vars(ds_in, alias_dict):
    rename_dict = {}

    for canonical, aliases in alias_dict.items():
        found = find_var_name(ds_in, aliases)

        if found is not None and found != canonical:
            rename_dict[found] = canonical

    if rename_dict:
        ds_in = ds_in.rename(rename_dict)

    return ds_in


def standardize_copernicus_dataset(path, alias_dict, source_label):
    """
    Abre, renombra variables y normaliza coords:
    time, lat, lon.
    También selecciona superficie si hay dimensión de profundidad.
    """
    path = Path(path)
    ds_in = open_dataset_robust(path)
    ds_in = rename_vars(ds_in, alias_dict)

    time_name = find_coord_name(ds_in, ["time", "valid_time"])
    lat_name = find_coord_name(ds_in, ["latitude", "lat"])
    lon_name = find_coord_name(ds_in, ["longitude", "lon"])

    if time_name is None:
        raise ValueError(f"{source_label} {path.name}: no se encontró coordenada temporal. Coords={list(ds_in.coords)}")

    if lat_name is None or lon_name is None:
        raise ValueError(
            f"{source_label} {path.name}: no se encontraron lat/lon. "
            f"Coords={list(ds_in.coords)} Dims={dict(ds_in.dims)}"
        )

    rename_coords = {}

    if time_name != "time":
        rename_coords[time_name] = "time"

    if lat_name != "lat":
        rename_coords[lat_name] = "lat"

    if lon_name != "lon":
        rename_coords[lon_name] = "lon"

    if rename_coords:
        ds_in = ds_in.rename(rename_coords)

    # Seleccionar primera capa de profundidad si existe.
    core_dims = {"time", "lat", "lon"}
    extra_dims = [d for d in ds_in.dims if d not in core_dims]

    for d in extra_dims:
        d_norm = normalize_col(d)
        if "DEPTH" in d_norm or "LEV" in d_norm or "Z" == d_norm or ds_in.sizes.get(d, 0) == 1:
            ds_in = ds_in.isel({d: 0})
        else:
            # Si queda una dimensión extra no esperada, usar primera posición y documentarlo.
            print(f"AVISO {path.name}: dimensión extra {d}; se usa índice 0.")
            ds_in = ds_in.isel({d: 0})

    # Longitudes -180..180.
    lon_std = standardize_longitudes_to_180(ds_in["lon"].values)
    ds_in = ds_in.assign_coords(lon=lon_std)
    ds_in = ds_in.sortby("lon")
    ds_in = ds_in.sortby("lat")

    ds_in = ds_in.sel(
        lat=slice(BBOX_CANARIAS["lat_min"], BBOX_CANARIAS["lat_max"]),
        lon=slice(BBOX_CANARIAS["lon_min"], BBOX_CANARIAS["lon_max"]),
    )

    if ds_in.sizes.get("lat", 0) == 0 or ds_in.sizes.get("lon", 0) == 0:
        raise ValueError(f"{source_label} {path.name}: recorte bbox vacío.")

    return ds_in


def get_years_in_dataset(ds_in):
    years = pd.to_datetime(ds_in["time"].values, utc=True, errors="coerce").year
    years = sorted(pd.Series(years).dropna().astype(int).unique().tolist())

    if MAX_YEARS_FOR_TEST is not None:
        years = years[:MAX_YEARS_FOR_TEST]

    return years


def dataset_to_flat_dataframe(ds_in, vars_to_use):
    available = [v for v in vars_to_use if v in ds_in.data_vars]

    if not available:
        return pd.DataFrame()

    df = ds_in[available].to_dataframe().reset_index()
    df = df.rename(columns={"time": "timestamp"})
    df["timestamp"] = ensure_utc(df["timestamp"])
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df = df.dropna(subset=["timestamp", "lat", "lon"]).copy()

    return df


def build_grid_zone_map(df_grid, prefix):
    unique_points = (
        df_grid[["lat", "lon"]]
        .drop_duplicates()
        .reset_index(drop=True)
        .copy()
    )

    unique_points["point_id"] = make_point_id(prefix, unique_points["lat"], unique_points["lon"])

    gdf_points = gpd.GeoDataFrame(
        unique_points,
        geometry=gpd.points_from_xy(unique_points["lon"], unique_points["lat"]),
        crs="EPSG:4326",
    )

    gdf_points_m = gdf_points.to_crs("EPSG:3857")

    nearest = gpd.sjoin_nearest(
        gdf_points_m,
        gdf_zones_m[["zona_id", "nombre_zona", "isla", "municipio", "geometry"]],
        how="left",
        distance_col="distance_to_zona_m",
    )

    nearest = (
        nearest
        .sort_values("distance_to_zona_m")
        .groupby("point_id", as_index=False)
        .first()
    )

    point_zone = pd.DataFrame(nearest.drop(columns="geometry", errors="ignore"))
    point_zone["distance_to_zona_km"] = point_zone["distance_to_zona_m"] / 1000

    point_zone = point_zone[
        [
            "point_id",
            "lat",
            "lon",
            "zona_id",
            "nombre_zona",
            "isla",
            "municipio",
            "distance_to_zona_km",
        ]
    ].copy()

    return point_zone

## Celda 7 — Flags de calidad

In [24]:
VARIABLE_RANGES = {
    # waves
    "hs": (0, 15),
    "hmax": (0, 25),
    "tp": (0, 35),
    "tm02": (0, 35),
    "wave_direction": (0, 360),
    "swell_height": (0, 15),
    "swell_period": (0, 35),
    "swell_direction": (0, 360),
    "wind_wave_height": (0, 15),
    "wind_wave_period": (0, 35),
    "stokes_drift": (0, 5),
    # physics
    "current_u": (-5, 5),
    "current_v": (-5, 5),
    "current_speed": (0, 5),
    "current_direction": (0, 360),
    "sea_surface_temperature": (5, 35),
    "sea_surface_salinity": (20, 45),
    "sea_level": (-3, 3),
}


def add_quality_flags(df, variable_ranges, interpolated_col=None):
    df = df.copy()

    interpolated_mask = None

    if interpolated_col is not None and interpolated_col in df.columns:
        interpolated_mask = df[interpolated_col].fillna(False).astype(bool)

    for col, (vmin, vmax) in variable_ranges.items():
        if col not in df.columns:
            continue

        flag_col = f"{col}_flag"
        df[flag_col] = 0

        if interpolated_mask is not None:
            df.loc[interpolated_mask & df[col].notna(), flag_col] = 3

        missing_mask = df[col].isna()
        outlier_mask = (~missing_mask) & ((df[col] < vmin) | (df[col] > vmax))

        # missing/outlier tienen prioridad sobre interpolated.
        df.loc[missing_mask, flag_col] = 1
        df.loc[outlier_mask, flag_col] = 2

        df[flag_col] = df[flag_col].astype("int8")

    return df

## Celda 8 — Transformaciones de waves → `ocean_hourly`

In [25]:
WAVE_RAW_CANONICAL = [
    "hs",
    "tp",
    "wave_direction",
    "swell_height",
    "swell_period",
    "swell_direction",
    "wind_wave_height",
    "wind_wave_period",
    "stokes_u",
    "stokes_v",
]

OCEAN_COLUMNS = [
    "timestamp",
    "zona_id",
    "copernicus_point_id",
    "lat",
    "lon",
    "source",
    "hs",
    "hmax",
    "tp",
    "tm02",
    "wave_direction",
    "swell_height",
    "swell_period",
    "swell_direction",
    "wind_wave_height",
    "wind_wave_period",
    "stokes_drift",
    "interpolated",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]


def make_ocean_chunk_from_waves(ds_year, filename):
    available = [v for v in WAVE_RAW_CANONICAL if v in ds_year.data_vars]

    if not available:
        return pd.DataFrame(columns=OCEAN_COLUMNS), {"available_wave_vars": []}

    original_times = pd.to_datetime(ds_year["time"].values, utc=True, errors="coerce")
    original_times = pd.Series(original_times).dropna().drop_duplicates().sort_values()

    # Resample a 1 hora si la frecuencia original es mayor que 1h.
    if len(original_times) >= 2:
        median_diff_h = original_times.diff().dropna().dt.total_seconds().median() / 3600
    else:
        median_diff_h = np.nan

    if pd.notna(median_diff_h) and median_diff_h > 1.1:
        ds_hourly = ds_year[available].resample(time="1h").interpolate("linear")
        interpolation_applied = True
    else:
        ds_hourly = ds_year[available]
        interpolation_applied = False

    df = dataset_to_flat_dataframe(ds_hourly, available)

    if df.empty:
        return pd.DataFrame(columns=OCEAN_COLUMNS), {
            "available_wave_vars": available,
            "interpolation_applied": interpolation_applied,
        }

    original_set = set(original_times.astype("datetime64[ns]"))
    df["interpolated"] = ~df["timestamp"].dt.tz_convert(None).astype("datetime64[ns]").isin(original_set)
    if not interpolation_applied:
        df["interpolated"] = False

    df["copernicus_point_id"] = make_point_id("COPW", df["lat"], df["lon"])

    point_zone = build_grid_zone_map(df[["lat", "lon"]].drop_duplicates(), prefix="COPW")

    df = df.merge(
        point_zone[
            [
                "point_id",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "distance_to_zona_km",
            ]
        ].rename(columns={"point_id": "copernicus_point_id"}),
        on="copernicus_point_id",
        how="left",
    )

    df["source"] = SOURCE_WAVES
    df["temporal_resolution"] = "hourly"
    df["year"] = df["timestamp"].dt.year.astype("Int64")

    if "stokes_u" in df.columns and "stokes_v" in df.columns:
        df["stokes_drift"] = np.sqrt(df["stokes_u"] ** 2 + df["stokes_v"] ** 2)
    else:
        df["stokes_drift"] = np.nan

    # No disponibles o no descargadas.
    df["hmax"] = np.nan
    df["tm02"] = np.nan

    if "wave_direction" in df.columns:
        df["wave_direction"] = df["wave_direction"] % 360

    if "swell_direction" in df.columns:
        df["swell_direction"] = df["swell_direction"] % 360

    for col in OCEAN_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df = df[OCEAN_COLUMNS].copy()

    df = (
        df
        .sort_values(["copernicus_point_id", "timestamp"])
        .drop_duplicates(subset=["copernicus_point_id", "timestamp", "source"], keep="first")
    )

    df = add_quality_flags(df, VARIABLE_RANGES, interpolated_col="interpolated")

    metadata = {
        "available_wave_vars": available,
        "interpolation_applied": interpolation_applied,
        "median_original_step_hours": median_diff_h,
        "unique_points": df["copernicus_point_id"].nunique(),
        "point_zone_rows": len(point_zone),
    }

    return df, metadata

parche

In [26]:
def make_ocean_chunk_from_waves(ds_year, filename):
    available = [v for v in WAVE_RAW_CANONICAL if v in ds_year.data_vars]

    if not available:
        return pd.DataFrame(columns=OCEAN_COLUMNS), {"available_wave_vars": []}

    # Fechas originales del producto Copernicus, normalmente cada 3h.
    original_times = pd.to_datetime(
        ds_year["time"].values,
        utc=True,
        errors="coerce",
    )

    original_times = pd.DatetimeIndex(original_times).dropna().unique().sort_values()

    if len(original_times) >= 2:
        median_diff_h = pd.Series(original_times).diff().dropna().dt.total_seconds().median() / 3600
    else:
        median_diff_h = np.nan

    # Resample a 1 hora si la frecuencia original es mayor que 1h.
    if pd.notna(median_diff_h) and median_diff_h > 1.1:
        ds_hourly = ds_year[available].resample(time="1h").interpolate("linear")
        interpolation_applied = True
    else:
        ds_hourly = ds_year[available]
        interpolation_applied = False

    df = dataset_to_flat_dataframe(ds_hourly, available)

    if df.empty:
        return pd.DataFrame(columns=OCEAN_COLUMNS), {
            "available_wave_vars": available,
            "interpolation_applied": interpolation_applied,
        }

    # Comparación robusta de timestamps usando enteros ns UTC.
    original_ns = set(original_times.asi8)

    df_timestamps = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="coerce",
    )

    df_ns = pd.DatetimeIndex(df_timestamps).asi8

    df["interpolated"] = ~pd.Series(df_ns, index=df.index).isin(original_ns)

    if not interpolation_applied:
        df["interpolated"] = False

    df["copernicus_point_id"] = make_point_id("COPW", df["lat"], df["lon"])

    point_zone = build_grid_zone_map(
        df[["lat", "lon"]].drop_duplicates(),
        prefix="COPW",
    )

    df = df.merge(
        point_zone[
            [
                "point_id",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "distance_to_zona_km",
            ]
        ].rename(columns={"point_id": "copernicus_point_id"}),
        on="copernicus_point_id",
        how="left",
    )

    df["source"] = SOURCE_WAVES
    df["temporal_resolution"] = "hourly"
    df["year"] = df["timestamp"].dt.year.astype("Int64")

    if "stokes_u" in df.columns and "stokes_v" in df.columns:
        df["stokes_drift"] = np.sqrt(df["stokes_u"] ** 2 + df["stokes_v"] ** 2)
    else:
        df["stokes_drift"] = np.nan

    # No disponibles o no descargadas.
    df["hmax"] = np.nan
    df["tm02"] = np.nan

    if "wave_direction" in df.columns:
        df["wave_direction"] = df["wave_direction"] % 360

    if "swell_direction" in df.columns:
        df["swell_direction"] = df["swell_direction"] % 360

    for col in OCEAN_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    df = df[OCEAN_COLUMNS].copy()

    df = (
        df
        .sort_values(["copernicus_point_id", "timestamp"])
        .drop_duplicates(
            subset=["copernicus_point_id", "timestamp", "source"],
            keep="first",
        )
    )

    df = add_quality_flags(
        df,
        VARIABLE_RANGES,
        interpolated_col="interpolated",
    )

    metadata = {
        "available_wave_vars": available,
        "interpolation_applied": interpolation_applied,
        "median_original_step_hours": median_diff_h,
        "unique_points": df["copernicus_point_id"].nunique(),
        "point_zone_rows": len(point_zone),
    }

    return df, metadata


print("Parche 8B cargado: make_ocean_chunk_from_waves corregida para timestamps UTC.")

Parche 8B cargado: make_ocean_chunk_from_waves corregida para timestamps UTC.


## Celda 9 — Transformaciones physics → `ocean_physics` y `tide_hourly`

In [27]:
PHYSICS_RAW_CANONICAL = [
    "current_u",
    "current_v",
    "sea_surface_temperature",
    "sea_surface_salinity",
    "zos",
]

PHYSICS_COLUMNS = [
    "timestamp",
    "zona_id",
    "copernicus_point_id",
    "lat",
    "lon",
    "source",
    "current_u",
    "current_v",
    "current_speed",
    "current_direction",
    "sea_surface_temperature",
    "sea_surface_salinity",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]

TIDE_ZOS_COLUMNS = [
    "timestamp",
    "station_id",
    "zona_id",
    "lat",
    "lon",
    "source",
    "sea_level",
    "astronomical_tide",
    "meteorological_residual",
    "tide_phase",
    "next_high_tide_time",
    "next_low_tide_time",
    "hours_to_high_tide",
    "hours_to_low_tide",
    "daily_tidal_range",
    "distance_to_zona_km",
    "temporal_resolution",
    "year",
    "isla",
]


def make_physics_and_zos_chunks(ds_year, filename):
    available = [v for v in PHYSICS_RAW_CANONICAL if v in ds_year.data_vars]

    if not available:
        return (
            pd.DataFrame(columns=PHYSICS_COLUMNS),
            pd.DataFrame(columns=TIDE_ZOS_COLUMNS),
            {"available_physics_vars": []},
        )

    df = dataset_to_flat_dataframe(ds_year, available)

    if df.empty:
        return (
            pd.DataFrame(columns=PHYSICS_COLUMNS),
            pd.DataFrame(columns=TIDE_ZOS_COLUMNS),
            {"available_physics_vars": available},
        )

    df["copernicus_point_id"] = make_point_id("COPP", df["lat"], df["lon"])

    point_zone = build_grid_zone_map(df[["lat", "lon"]].drop_duplicates(), prefix="COPP")

    df = df.merge(
        point_zone[
            [
                "point_id",
                "zona_id",
                "nombre_zona",
                "isla",
                "municipio",
                "distance_to_zona_km",
            ]
        ].rename(columns={"point_id": "copernicus_point_id"}),
        on="copernicus_point_id",
        how="left",
    )

    df["year"] = df["timestamp"].dt.year.astype("Int64")

    # Ocean physics.
    physics = df.copy()
    physics["source"] = SOURCE_PHYSICS
    physics["temporal_resolution"] = "daily"

    if {"current_u", "current_v"}.issubset(physics.columns):
        physics["current_speed"] = current_speed_from_uv(physics["current_u"], physics["current_v"])
        physics["current_direction"] = current_direction_from_uv(physics["current_u"], physics["current_v"])
    else:
        physics["current_speed"] = np.nan
        physics["current_direction"] = np.nan

    for col in PHYSICS_COLUMNS:
        if col not in physics.columns:
            physics[col] = np.nan

    physics = physics[PHYSICS_COLUMNS].copy()

    physics = (
        physics
        .sort_values(["copernicus_point_id", "timestamp"])
        .drop_duplicates(subset=["copernicus_point_id", "timestamp", "source"], keep="first")
    )

    physics = add_quality_flags(physics, VARIABLE_RANGES)

    # ZOS → tide_hourly como sea_level diario si existe.
    if "zos" in df.columns:
        zos = df.copy()
        zos["station_id"] = zos["copernicus_point_id"]
        zos["source"] = SOURCE_ZOS
        zos["sea_level"] = pd.to_numeric(zos["zos"], errors="coerce")
        zos["astronomical_tide"] = np.nan
        zos["meteorological_residual"] = np.nan
        zos["tide_phase"] = "not_applicable_daily_zos"
        zos["next_high_tide_time"] = pd.NaT
        zos["next_low_tide_time"] = pd.NaT
        zos["hours_to_high_tide"] = np.nan
        zos["hours_to_low_tide"] = np.nan
        zos["daily_tidal_range"] = np.nan
        zos["temporal_resolution"] = "daily"

        for col in TIDE_ZOS_COLUMNS:
            if col not in zos.columns:
                zos[col] = np.nan

        zos = zos[TIDE_ZOS_COLUMNS].copy()

        zos = (
            zos
            .sort_values(["station_id", "timestamp"])
            .drop_duplicates(subset=["station_id", "timestamp", "source"], keep="first")
        )

        zos = add_quality_flags(zos, VARIABLE_RANGES)

    else:
        zos = pd.DataFrame(columns=TIDE_ZOS_COLUMNS)

    metadata = {
        "available_physics_vars": available,
        "unique_points": df["copernicus_point_id"].nunique(),
        "point_zone_rows": len(point_zone),
        "zos_rows": len(zos),
    }

    return physics, zos, metadata

## Celda 10 — Borrar particiones Copernicus antiguas

In [28]:
remove_existing_source_partition(OUT_OCEAN_DIR, SOURCE_WAVES)
remove_existing_source_partition(OUT_PHYSICS_DIR, SOURCE_PHYSICS)
remove_existing_source_partition(OUT_TIDE_DIR, SOURCE_ZOS)

print("Particiones Copernicus antiguas eliminadas si existían.")

Eliminada partición antigua: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_hourly/source=COPERNICUS_MARINE_WAVES
Particiones Copernicus antiguas eliminadas si existían.


## Celda 11 — Procesar Copernicus Marine waves

In [29]:
wave_file_summaries = []
wave_errors = []
wave_point_maps = []

for p in tqdm(wave_files, desc="Procesando Copernicus waves"):
    ds_wave = None

    try:
        ds_wave = standardize_copernicus_dataset(
            p,
            alias_dict=WAVE_VAR_ALIASES,
            source_label="waves",
        )

        years = get_years_in_dataset(ds_wave)

        for year in years:
            ds_year = ds_wave.sel(time=slice(f"{year}-01-01", f"{year}-12-31T23:59:59"))

            if ds_year.sizes.get("time", 0) == 0:
                continue

            ocean_chunk, meta = make_ocean_chunk_from_waves(ds_year, p.name)

            validate_timestamp_range(ocean_chunk, "ocean_hourly", p.name)

            if not ocean_chunk.empty:
                if ocean_chunk["zona_id"].isna().any():
                    raise ValueError(f"{p.name} {year}: ocean_chunk tiene zona_id nulos.")

                write_partitioned_parquet(ocean_chunk, OUT_OCEAN_DIR)

                point_map = (
                    ocean_chunk[
                        [
                            "copernicus_point_id",
                            "lat",
                            "lon",
                            "zona_id",
                            "isla",
                            "distance_to_zona_km",
                        ]
                    ]
                    .drop_duplicates(subset=["copernicus_point_id"])
                    .copy()
                )
                point_map["filename"] = p.name
                wave_point_maps.append(point_map)

            wave_file_summaries.append(
                {
                    "filename": p.name,
                    "year": year,
                    "rows": len(ocean_chunk),
                    "timestamp_min": ocean_chunk["timestamp"].min() if len(ocean_chunk) else pd.NaT,
                    "timestamp_max": ocean_chunk["timestamp"].max() if len(ocean_chunk) else pd.NaT,
                    "available_wave_vars": json.dumps(meta.get("available_wave_vars", []), ensure_ascii=False),
                    "interpolation_applied": meta.get("interpolation_applied"),
                    "median_original_step_hours": meta.get("median_original_step_hours"),
                    "unique_points": meta.get("unique_points"),
                    "hs_missing_pct": missing_pct(ocean_chunk, "hs"),
                    "tp_missing_pct": missing_pct(ocean_chunk, "tp"),
                    "wave_direction_missing_pct": missing_pct(ocean_chunk, "wave_direction"),
                    "interpolated_pct": float(ocean_chunk["interpolated"].mean() * 100) if len(ocean_chunk) and "interpolated" in ocean_chunk else np.nan,
                }
            )

            del ocean_chunk, ds_year
            gc.collect()

    except Exception as e:
        wave_errors.append(
            {
                "filename": p.name,
                "path": str(p),
                "error": repr(e),
            }
        )

    finally:
        if ds_wave is not None:
            ds_wave.close()

wave_file_summary_df = pd.DataFrame(wave_file_summaries)
wave_errors_df = pd.DataFrame(wave_errors)

print("Wave chunks procesados:", len(wave_file_summary_df))
print("Errores waves:", len(wave_errors_df))

display(wave_file_summary_df.head())
display(wave_errors_df)

wave_file_summary_df.to_csv(QC_DIR / "quality_copernicus_waves_file_summary.csv", index=False)
wave_errors_df.to_csv(QC_DIR / "quality_copernicus_waves_errors.csv", index=False)

if wave_point_maps:
    wave_point_to_zone = (
        pd.concat(wave_point_maps, ignore_index=True)
        .sort_values(["copernicus_point_id", "distance_to_zona_km"])
        .drop_duplicates(subset=["copernicus_point_id"], keep="first")
        .reset_index(drop=True)
    )
else:
    wave_point_to_zone = pd.DataFrame()

wave_point_to_zone.to_csv(META_DIR / "copernicus_waves_point_to_zone.csv", index=False)

if len(wave_errors_df):
    raise ValueError("Hay errores procesando Copernicus waves. Revisar quality_copernicus_waves_errors.csv.")

Procesando Copernicus waves:   0%|          | 0/3 [00:00<?, ?it/s]

Wave chunks procesados: 26
Errores waves: 0


,filename,year,rows,timestamp_min,timestamp_max,available_wave_vars,interpolation_applied,median_original_step_hours,unique_points,hs_missing_pct,tp_missing_pct,wave_direction_missing_pct,interpolated_pct
0,cmems_mod_glo_wav_my_0.2deg_PT3H-i_17781739328...,2001,2594889,2001-01-01 21:00:00+00:00,2001-12-31 21:00:00+00:00,"[""hs"", ""tp"", ""wave_direction"", ""swell_height"",...",True,3.0,297,1.683502,1.683502,1.683502,66.659036
1,cmems_mod_glo_wav_my_0.2deg_PT3H-i_17781739328...,2002,2601126,2002-01-01 00:00:00+00:00,2002-12-31 21:00:00+00:00,"[""hs"", ""tp"", ""wave_direction"", ""swell_height"",...",True,3.0,297,1.683502,1.683502,1.683502,66.659055
2,cmems_mod_glo_wav_my_0.2deg_PT3H-i_17781739328...,2003,2601126,2003-01-01 00:00:00+00:00,2003-12-31 21:00:00+00:00,"[""hs"", ""tp"", ""wave_direction"", ""swell_height"",...",True,3.0,297,1.683502,1.683502,1.683502,66.659055
3,cmems_mod_glo_wav_my_0.2deg_PT3H-i_17781739328...,2004,2608254,2004-01-01 00:00:00+00:00,2004-12-31 21:00:00+00:00,"[""hs"", ""tp"", ""wave_direction"", ""swell_height"",...",True,3.0,297,1.683502,1.683502,1.683502,66.659075
4,cmems_mod_glo_wav_my_0.2deg_PT3H-i_17781739328...,2005,2601126,2005-01-01 00:00:00+00:00,2005-12-31 21:00:00+00:00,"[""hs"", ""tp"", ""wave_direction"", ""swell_height"",...",True,3.0,297,1.683502,1.683502,1.683502,66.659055


""


## Celda 12 — Procesar Copernicus Marine physics

In [30]:
physics_file_summaries = []
physics_errors = []
physics_point_maps = []

for p in tqdm(physics_files, desc="Procesando Copernicus physics"):
    ds_phys = None

    try:
        ds_phys = standardize_copernicus_dataset(
            p,
            alias_dict=PHYSICS_VAR_ALIASES,
            source_label="physics",
        )

        years = get_years_in_dataset(ds_phys)

        for year in years:
            ds_year = ds_phys.sel(time=slice(f"{year}-01-01", f"{year}-12-31T23:59:59"))

            if ds_year.sizes.get("time", 0) == 0:
                continue

            physics_chunk, zos_chunk, meta = make_physics_and_zos_chunks(ds_year, p.name)

            validate_timestamp_range(physics_chunk, "ocean_physics", p.name)
            validate_timestamp_range(zos_chunk, "tide_hourly_zos", p.name)

            if not physics_chunk.empty:
                if physics_chunk["zona_id"].isna().any():
                    raise ValueError(f"{p.name} {year}: physics_chunk tiene zona_id nulos.")

                write_partitioned_parquet(physics_chunk, OUT_PHYSICS_DIR)

                point_map = (
                    physics_chunk[
                        [
                            "copernicus_point_id",
                            "lat",
                            "lon",
                            "zona_id",
                            "isla",
                            "distance_to_zona_km",
                        ]
                    ]
                    .drop_duplicates(subset=["copernicus_point_id"])
                    .copy()
                )
                point_map["filename"] = p.name
                physics_point_maps.append(point_map)

            if not zos_chunk.empty:
                if zos_chunk["zona_id"].isna().any():
                    raise ValueError(f"{p.name} {year}: zos_chunk tiene zona_id nulos.")

                write_partitioned_parquet(zos_chunk, OUT_TIDE_DIR)

            physics_file_summaries.append(
                {
                    "filename": p.name,
                    "year": year,
                    "physics_rows": len(physics_chunk),
                    "zos_rows": len(zos_chunk),
                    "timestamp_min": physics_chunk["timestamp"].min() if len(physics_chunk) else pd.NaT,
                    "timestamp_max": physics_chunk["timestamp"].max() if len(physics_chunk) else pd.NaT,
                    "available_physics_vars": json.dumps(meta.get("available_physics_vars", []), ensure_ascii=False),
                    "unique_points": meta.get("unique_points"),
                    "current_speed_missing_pct": missing_pct(physics_chunk, "current_speed"),
                    "temperature_missing_pct": missing_pct(physics_chunk, "sea_surface_temperature"),
                    "salinity_missing_pct": missing_pct(physics_chunk, "sea_surface_salinity"),
                    "zos_missing_pct": missing_pct(zos_chunk, "sea_level") if len(zos_chunk) else np.nan,
                }
            )

            del physics_chunk, zos_chunk, ds_year
            gc.collect()

    except Exception as e:
        physics_errors.append(
            {
                "filename": p.name,
                "path": str(p),
                "error": repr(e),
            }
        )

    finally:
        if ds_phys is not None:
            ds_phys.close()

physics_file_summary_df = pd.DataFrame(physics_file_summaries)
physics_errors_df = pd.DataFrame(physics_errors)

print("Physics chunks procesados:", len(physics_file_summary_df))
print("Errores physics:", len(physics_errors_df))

display(physics_file_summary_df.head())
display(physics_errors_df)

physics_file_summary_df.to_csv(QC_DIR / "quality_copernicus_physics_file_summary.csv", index=False)
physics_errors_df.to_csv(QC_DIR / "quality_copernicus_physics_errors.csv", index=False)

if physics_point_maps:
    physics_point_to_zone = (
        pd.concat(physics_point_maps, ignore_index=True)
        .sort_values(["copernicus_point_id", "distance_to_zona_km"])
        .drop_duplicates(subset=["copernicus_point_id"], keep="first")
        .reset_index(drop=True)
    )
else:
    physics_point_to_zone = pd.DataFrame()

physics_point_to_zone.to_csv(META_DIR / "copernicus_physics_point_to_zone.csv", index=False)

if len(physics_errors_df):
    raise ValueError("Hay errores procesando Copernicus physics. Revisar quality_copernicus_physics_errors.csv.")

Procesando Copernicus physics:   0%|          | 0/1 [00:00<?, ?it/s]

Physics chunks procesados: 25
Errores physics: 0


,filename,year,physics_rows,zos_rows,timestamp_min,timestamp_max,available_physics_vars,unique_points,current_speed_missing_pct,temperature_missing_pct,salinity_missing_pct,zos_missing_pct
0,cmems_mod_glo_phy_my_0.083deg_P1D-m_1778177336...,2001,746790,0,2001-01-01 00:00:00+00:00,2001-12-31 00:00:00+00:00,"[""sea_surface_temperature"", ""sea_surface_salin...",2046,100.0,4.985337,4.985337,NaN
1,cmems_mod_glo_phy_my_0.083deg_P1D-m_1778177336...,2002,746790,0,2002-01-01 00:00:00+00:00,2002-12-31 00:00:00+00:00,"[""sea_surface_temperature"", ""sea_surface_salin...",2046,100.0,4.985337,4.985337,NaN
2,cmems_mod_glo_phy_my_0.083deg_P1D-m_1778177336...,2003,746790,0,2003-01-01 00:00:00+00:00,2003-12-31 00:00:00+00:00,"[""sea_surface_temperature"", ""sea_surface_salin...",2046,100.0,4.985337,4.985337,NaN
3,cmems_mod_glo_phy_my_0.083deg_P1D-m_1778177336...,2004,748836,0,2004-01-01 00:00:00+00:00,2004-12-31 00:00:00+00:00,"[""sea_surface_temperature"", ""sea_surface_salin...",2046,100.0,4.985337,4.985337,NaN
4,cmems_mod_glo_phy_my_0.083deg_P1D-m_1778177336...,2005,746790,0,2005-01-01 00:00:00+00:00,2005-12-31 00:00:00+00:00,"[""sea_surface_temperature"", ""sea_surface_salin...",2046,100.0,4.985337,4.985337,NaN


""


## Celda 13 — Resúmenes globales y validación de outputs

In [31]:
ocean_count, ocean_sample = dataset_count_and_sample(OUT_OCEAN_DIR, SOURCE_WAVES)
physics_count, physics_sample = dataset_count_and_sample(OUT_PHYSICS_DIR, SOURCE_PHYSICS)
zos_count, zos_sample = dataset_count_and_sample(OUT_TIDE_DIR, SOURCE_ZOS)

print("Filas ocean_hourly Copernicus waves:", ocean_count)
print("Filas ocean_physics Copernicus physics:", physics_count)
print("Filas tide_hourly Copernicus ZOS:", zos_count)

if len(ocean_sample):
    print("Sample ocean_hourly waves:")
    display(ocean_sample)

if len(physics_sample):
    print("Sample ocean_physics:")
    display(physics_sample)

if len(zos_sample):
    print("Sample tide_hourly ZOS:")
    display(zos_sample)

copernicus_global_summary = pd.DataFrame(
    [
        {
            "table": "ocean_hourly",
            "source": SOURCE_WAVES,
            "rows": ocean_count,
            "chunks_processed": len(wave_file_summary_df),
            "errors": len(wave_errors_df),
            "timestamp_min": wave_file_summary_df["timestamp_min"].min() if len(wave_file_summary_df) else pd.NaT,
            "timestamp_max": wave_file_summary_df["timestamp_max"].max() if len(wave_file_summary_df) else pd.NaT,
            "mean_hs_missing_pct": wave_file_summary_df["hs_missing_pct"].mean() if len(wave_file_summary_df) else np.nan,
            "mean_tp_missing_pct": wave_file_summary_df["tp_missing_pct"].mean() if len(wave_file_summary_df) else np.nan,
        },
        {
            "table": "ocean_physics",
            "source": SOURCE_PHYSICS,
            "rows": physics_count,
            "chunks_processed": len(physics_file_summary_df),
            "errors": len(physics_errors_df),
            "timestamp_min": physics_file_summary_df["timestamp_min"].min() if len(physics_file_summary_df) else pd.NaT,
            "timestamp_max": physics_file_summary_df["timestamp_max"].max() if len(physics_file_summary_df) else pd.NaT,
            "mean_current_speed_missing_pct": physics_file_summary_df["current_speed_missing_pct"].mean() if len(physics_file_summary_df) else np.nan,
            "mean_temperature_missing_pct": physics_file_summary_df["temperature_missing_pct"].mean() if len(physics_file_summary_df) else np.nan,
            "mean_salinity_missing_pct": physics_file_summary_df["salinity_missing_pct"].mean() if len(physics_file_summary_df) else np.nan,
        },
        {
            "table": "tide_hourly",
            "source": SOURCE_ZOS,
            "rows": zos_count,
            "chunks_processed": len(physics_file_summary_df),
            "errors": len(physics_errors_df),
            "timestamp_min": physics_file_summary_df["timestamp_min"].min() if len(physics_file_summary_df) else pd.NaT,
            "timestamp_max": physics_file_summary_df["timestamp_max"].max() if len(physics_file_summary_df) else pd.NaT,
            "mean_zos_missing_pct": physics_file_summary_df["zos_missing_pct"].mean() if len(physics_file_summary_df) else np.nan,
        },
    ]
)

display(copernicus_global_summary)

copernicus_global_summary.to_csv(QC_DIR / "quality_copernicus_global_summary.csv", index=False)

Filas ocean_hourly Copernicus waves: 65472165
Filas ocean_physics Copernicus physics: 18682026
Filas tide_hourly Copernicus ZOS: 0
Sample ocean_hourly waves:


,timestamp,zona_id,copernicus_point_id,lat,lon,hs,hmax,tp,tm02,wave_direction,...,wave_direction_flag,swell_height_flag,swell_period_flag,swell_direction_flag,wind_wave_height_flag,wind_wave_period_flag,stokes_drift_flag,source,year,isla
0,2001-01-01 21:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPW_29.4_-13.2,29.400002,-13.200012,2.590000,NaN,19.990000,NaN,314.869995,...,0,0,1,0,0,1,0,COPERNICUS_MARINE_WAVES,2001,Alegranza
1,2001-01-01 22:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPW_29.4_-13.2,29.400002,-13.200012,2.806667,NaN,19.890000,NaN,315.349996,...,3,3,1,3,3,1,3,COPERNICUS_MARINE_WAVES,2001,Alegranza
2,2001-01-01 23:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPW_29.4_-13.2,29.400002,-13.200012,3.023333,NaN,19.790000,NaN,315.829997,...,3,3,1,3,3,1,3,COPERNICUS_MARINE_WAVES,2001,Alegranza
3,2001-01-02 00:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPW_29.4_-13.2,29.400002,-13.200012,3.240000,NaN,19.690001,NaN,316.309998,...,0,0,1,0,0,1,0,COPERNICUS_MARINE_WAVES,2001,Alegranza
4,2001-01-02 01:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPW_29.4_-13.2,29.400002,-13.200012,3.413333,NaN,19.486667,NaN,316.710002,...,3,3,1,3,3,1,3,COPERNICUS_MARINE_WAVES,2001,Alegranza


Sample ocean_physics:


,timestamp,zona_id,copernicus_point_id,lat,lon,current_u,current_v,current_speed,current_direction,sea_surface_temperature,...,temporal_resolution,current_u_flag,current_v_flag,current_speed_flag,current_direction_flag,sea_surface_temperature_flag,sea_surface_salinity_flag,source,year,isla
0,2001-01-01 00:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPP_29.3333_-13.5,29.333334,-13.499985,NaN,NaN,NaN,NaN,NaN,...,daily,1,1,1,1,1,1,COPERNICUS_MARINE_PHYSICS,2001,Alegranza
1,2001-01-02 00:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPP_29.3333_-13.5,29.333334,-13.499985,NaN,NaN,NaN,NaN,NaN,...,daily,1,1,1,1,1,1,COPERNICUS_MARINE_PHYSICS,2001,Alegranza
2,2001-01-03 00:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPP_29.3333_-13.5,29.333334,-13.499985,NaN,NaN,NaN,NaN,NaN,...,daily,1,1,1,1,1,1,COPERNICUS_MARINE_PHYSICS,2001,Alegranza
3,2001-01-04 00:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPP_29.3333_-13.5,29.333334,-13.499985,NaN,NaN,NaN,NaN,NaN,...,daily,1,1,1,1,1,1,COPERNICUS_MARINE_PHYSICS,2001,Alegranza
4,2001-01-05 00:00:00+00:00,CAN_LZ_ISLA_DE_LA_ALEGRANZA,COPP_29.3333_-13.5,29.333334,-13.499985,NaN,NaN,NaN,NaN,NaN,...,daily,1,1,1,1,1,1,COPERNICUS_MARINE_PHYSICS,2001,Alegranza


,table,source,rows,chunks_processed,errors,timestamp_min,timestamp_max,mean_hs_missing_pct,mean_tp_missing_pct,mean_current_speed_missing_pct,mean_temperature_missing_pct,mean_salinity_missing_pct,mean_zos_missing_pct
0,ocean_hourly,COPERNICUS_MARINE_WAVES,65472165,26,0,2001-01-01 21:00:00+00:00,2026-02-28 21:00:00+00:00,1.683502,1.683502,NaN,NaN,NaN,NaN
1,ocean_physics,COPERNICUS_MARINE_PHYSICS,18682026,25,0,2001-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,NaN,NaN,100.0,4.985337,4.985337,NaN
2,tide_hourly,COPERNICUS_MARINE_PHYSICS_ZOS,0,25,0,2001-01-01 00:00:00+00:00,2025-12-31 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN


## Celda 14 — Validaciones finales

In [32]:
# Validación waves.
if wave_files and ocean_count == 0:
    raise ValueError("Hay archivos Copernicus waves, pero no se generó ocean_hourly.")

if physics_files and physics_count == 0:
    raise ValueError("Hay archivos Copernicus physics, pero no se generó ocean_physics.")

if wave_files and len(wave_file_summary_df):
    bad_waves = wave_file_summary_df[
        (wave_file_summary_df["rows"] > 0)
        & (
            (wave_file_summary_df["hs_missing_pct"] > 20)
            | (wave_file_summary_df["tp_missing_pct"] > 20)
            | (wave_file_summary_df["wave_direction_missing_pct"] > 20)
        )
    ]

    if len(bad_waves):
        print("Chunks waves con demasiados nulos en variables clave:")
        display(bad_waves)
        raise ValueError("Hay chunks waves con demasiados nulos en hs/tp/wave_direction.")

if physics_files and len(physics_file_summary_df):
    bad_physics = physics_file_summary_df[
        (physics_file_summary_df["physics_rows"] > 0)
        & (
            (physics_file_summary_df["temperature_missing_pct"] > 20)
            | (physics_file_summary_df["salinity_missing_pct"] > 20)
        )
    ]

    if len(bad_physics):
        print("Chunks physics con demasiados nulos en temperatura/salinidad:")
        display(bad_physics)
        raise ValueError("Hay chunks physics con demasiados nulos en variables clave.")

# current_speed puede faltar si la descarga no trae uo/vo; avisar, no bloquear.
if physics_files and len(physics_file_summary_df):
    current_missing_mean = physics_file_summary_df["current_speed_missing_pct"].mean()
    if pd.notna(current_missing_mean) and current_missing_mean > 20:
        print(
            "AVISO: current_speed tiene muchos nulos. "
            "Puede significar que la descarga physics no contiene uo/vo."
        )

print("Validación final Copernicus Marine superada.")

AVISO: current_speed tiene muchos nulos. Puede significar que la descarga physics no contiene uo/vo.
Validación final Copernicus Marine superada.


## Celda 15 — Listado de salidas generadas

In [33]:
print("Parquet Copernicus generado en:")
print("-", OUT_OCEAN_DIR / f"source={SOURCE_WAVES}")
print("-", OUT_PHYSICS_DIR / f"source={SOURCE_PHYSICS}")
print("-", OUT_TIDE_DIR / f"source={SOURCE_ZOS}")

print("\nReportes de calidad Copernicus:")
for p in sorted(QC_DIR.glob("quality_copernicus*.csv")):
    print("-", p)

print("\nMetadatos Copernicus:")
for p in sorted(META_DIR.glob("copernicus*.csv")):
    print("-", p)

Parquet Copernicus generado en:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_hourly/source=COPERNICUS_MARINE_WAVES
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/ocean_physics/source=COPERNICUS_MARINE_PHYSICS
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/tide_hourly/source=COPERNICUS_MARINE_PHYSICS_ZOS

Reportes de calidad Copernicus:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_copernicus_global_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_copernicus_physics_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_copernicus_physics_file_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_copernicus_waves_errors.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/quality_copernicus_waves_file_summary.csv

Metadatos Coper

## Resultado esperado

Al terminar deberían existir:

```text
silver/ocean_hourly/source=COPERNICUS_MARINE_WAVES/year=YYYY/isla=.../*.parquet
silver/ocean_physics/source=COPERNICUS_MARINE_PHYSICS/year=YYYY/isla=.../*.parquet
silver/tide_hourly/source=COPERNICUS_MARINE_PHYSICS_ZOS/year=YYYY/isla=.../*.parquet

silver/_quality_reports/quality_copernicus_global_summary.csv
silver/_quality_reports/quality_copernicus_waves_file_summary.csv
silver/_quality_reports/quality_copernicus_physics_file_summary.csv
silver/_metadata/copernicus_waves_point_to_zone.csv
silver/_metadata/copernicus_physics_point_to_zone.csv
```

Comprueba especialmente:

```text
Filas ocean_hourly Copernicus waves > 0
Filas ocean_physics Copernicus physics > 0
Validación final Copernicus Marine superada
```

Si `tide_hourly` ZOS tiene filas, quedará como nivel diario de mar derivado de `zos`, no como marea horaria astronómica.